# Lecture 12 - Personalizing the tools

**Lecture Topic**: Making an agent your own (project memory, skills, sub-agents, and a word on fine-tuning), then cost and data constraints. Last Module B lecture before the midterm.

## Announcements

- **The midterm is on Monday, Oct 26, in class.** Closed-AI, no laptops. It covers Modules A and B: reading and debugging NumPy / pandas / xarray code, and how agents work.
- We'll spend the first few minutes on some example questions.
- ClassBuzz poll for today: `LNMQ`

## Recap: Module B so far

 - How an LLM works: tokens, sampling, knowledge cutoffs, the context window
 - The agent loop: the model asks for a tool, the harness runs it
 - The workflow: requirements → spec → plan → implement → verify
 - Context management, verifiers (tests), and security (prompt injection, permissions)

Today:
 - A few example midterm questions
 - Personalizing the harness: project memory, skills, sub-agents, fine-tuning
 - Cost and data constraints

# Midterm warm-up

The real exam is on paper and closed-AI. Here are three questions in the style you'll see. Have a go before the answer.

### Example 1 (reading code)

```python
import numpy as np
(np.ones((3, 1)) + np.ones(4)).shape
```

What is the shape of the result?

 - A. `(3,)`
 - B. `(4,)`
 - C. `(3, 4)`
 - D. An error: the shapes don't match

In [ ]:
import numpy as np
(np.ones((3, 1)) + np.ones(4)).shape

**C, `(3, 4)`.** Broadcasting lines the shapes up from the right: `(3, 1)` and `(4,)` become `(3, 1)` and `(1, 4)`, and each size-1 axis is stretched. Knowing broadcasting is how you predict (and catch) the shape an agent's array code produces.

### Example 2 (debugging)

```python
import pandas as pd
df = pd.DataFrame({"zone": [1, 1, None, 2], "fare": [10, 20, 30, 40]})
df.groupby("zone").size().sum()
```

The DataFrame has 4 rows. What does this print?

 - A. 4
 - B. 3
 - C. 2
 - D. An error

In [ ]:
import pandas as pd
df = pd.DataFrame({"zone": [1, 1, None, 2], "fare": [10, 20, 30, 40]})
df.groupby("zone").size().sum()

**B, 3.** `groupby` silently drops rows whose key is missing, so the row with `zone = None` vanishes from the counts. This is exactly the kind of quiet row loss you learned to check for with `merge(validate=...)` and row-count checks.

### Example 3 (how agents work)

You have a 30-turn conversation with a coding agent. Why does the *input* token cost grow faster than 30× the cost of a single turn?

 - A. The model gets slower as it tires
 - B. Each turn re-sends the whole conversation so far, so total input grows with the square of the turns
 - C. The context window shrinks each turn
 - D. It doesn't; cost is flat because of caching

**B.** The model has no memory, so every turn resends everything before it: turn *k* sends ~*k* turns' worth of tokens, and 1 + 2 + ... + 30 grows like 30². Caching makes the repeated prefix cheaper but doesn't change the growth. (A short-answer version might ask you to name two things that bound it: compaction and starting fresh.)

That's the flavour. The exam has more of these plus a couple of short-answer and one "critique this agent transcript" question. Bring your understanding of Modules A and B, not memorised syntax.

# Personalizing the tools

We've treated the harness as fixed. But a lot of what makes someone productive with an agent is *shaping* it to their project and their habits. There are broadly four ways to do this, from everyday to specialist:

| Lever | What it is | Effort |
|---|---|---|
| **Project memory** | A file the agent reads every session (`CLAUDE.md` / `AGENTS.md`) | Minutes |
| **Skills** | Reusable instructions for a recurring task, pulled in when relevant | An afternoon |
| **Sub-agents** | A separate agent with its own context does a sub-task | A prompt |
| **Fine-tuning** | Actually retraining a model's weights on your examples | Days, and a dataset |

Start with the cheapest one that works. Most of the value is in the top two.

## Project memory: `CLAUDE.md` / `AGENTS.md`

You've seen this all through Module B. It's loaded into *every* session, so it's where you put the things you'd otherwise retype constantly:

 - How to run things (the Python path, the test command)
 - Where things live (data folders, the spec)
 - Conventions and gotchas specific to your project

Keep it short and always-relevant. A bloated memory file is just context rot you've opted into. `AGENTS.md` is the same idea as a cross-tool standard that most agents now read, so prefer it if your team uses several tools.

## Skills

A **skill** is a short document that teaches the agent how to do a *recurring* task your way. It sits in your project, and the agent pulls it into context only when it's relevant, so it doesn't clutter every session.

Here's one that encodes the empirical-scaling procedure we'll formalise in Module C:

```markdown
---
name: scaling-check
description: Measure how a Python function's runtime grows with input size and
  report the empirical order of growth. Use whenever asked how something scales,
  whether it is fast enough, or to compare two implementations.
---

# Scaling check

When asked how a function scales, do this and nothing more:

1. Time the function on geometrically increasing input sizes (10k, 100k, 1M rows)
   by repeating the sample DataFrame, using `time.perf_counter`.
2. Fit a line to log(time) vs log(n); report the slope as the empirical exponent
   (slope ~1 is linear, ~2 is quadratic).
3. Print a small table of n and seconds, then one sentence: the exponent and
   whether it meets any stated budget.

Do not change the function. Keep the whole script under 25 lines.
```

The `description` is the important part: it's how the agent decides *when* to reach for the skill.

### The skill in action

I put that skill in a project (as `.claude/skills/scaling-check/SKILL.md`) with a `trips_per_zone` function, and simply asked Claude Code *"How does trips_per_zone scale?"* Here's what it did:

In [ ]:
import json

skill_log = [json.loads(line) for line in open('../data/claude_code_session_skill.jsonl')]

for event in skill_log:
    if event['type'] == 'assistant':
        for block in event['message']['content']:
            if block['type'] == 'tool_use':
                arg = block['input'].get('skill') or block['input'].get('command', '')
                print(f"-> {block['name']}: {arg[:70]}")
    elif event['type'] == 'system' and event.get('subtype') == 'permission_denied':
        print(f"   PERMISSION DENIED: {event['message'][:80]}")

It recognised the task, loaded the `scaling-check` skill, and followed the recipe: it timed the function at 10k / 100k / 1M rows and reported the growth, without me re-explaining any of it. (Note the permission boundary again: it tried to write a scratch file to `/tmp`, got blocked, and ran the code inline instead.)

In [ ]:
print(skill_log[-1]['result'])

The payoff: a procedure you'd otherwise re-describe every time becomes one line, and it runs the same way for everyone on the team. Good skill candidates are anything repetitive and particular: "release a notebook the way we do", "write tests in our house style", "check a data file against our schema".

## Sub-agents

Recall the context problem from Lecture 10: a long, cluttered context makes the agent worse. A **sub-agent** is a fresh agent the main one spins up to do a focused sub-task in *its own* context, then report back just a short summary.

What that buys you:

 - **Keeps the main context clean.** A sub-agent can read 40 files to answer "where is X configured?" and return one paragraph; the 40 files never touch your main conversation.
 - **Parallelism.** Several sub-agents can work at once, e.g. review a change for bugs, for style, and for performance separately, then combine.
 - **Specialisation.** You can give a sub-agent a narrow brief and only the tools it needs (least privilege, from the security lecture).

### A pattern you'll use

> *"Review this pull request from three angles, one sub-agent each: (1) correctness bugs, (2) missing tests, (3) scaling problems. Give me the findings grouped by angle."*

Each sub-agent sees only the diff and its own instructions, so none is distracted by the others' reasoning.

### That pattern, recorded

I put a small function up for review (`pairs.py`, meant to return the busiest pickup→dropoff pairs) with a planted bug, no tests, and a scaling problem, and asked Claude Code exactly that:

> *"Review pairs.py from three angles, using a separate sub-agent for each: (1) correctness bugs, (2) missing tests, (3) scaling to the full ~7M-row files."*

In [ ]:
import json

sub_log = [json.loads(line) for line in open('../data/claude_code_session_subagents.jsonl')]

for event in sub_log:
    if event['type'] == 'assistant':
        for block in event['message']['content']:
            if block['type'] == 'tool_use' and block['name'] in ('Agent', 'Task'):
                print(f"dispatched sub-agent: {block['input'].get('description')}")
    elif event['type'] == 'system' and event.get('subtype') == 'permission_denied':
        print(f"   (a sub-agent was blocked: {event['message'][:60]}...)")

Three sub-agents ran in their own contexts, one per angle, and the main agent waited and combined them. Each found its planted problem:

In [ ]:
print(sub_log[-1]['result'])

Notice the correctness sub-agent caught the real bug (the sort is ascending, so it returns the *least* busy pairs), the test sub-agent proposed the test that would have caught it, and the scaling sub-agent flagged the `iterrows` + quadratic `seen` lookup. None of their intermediate work, reading the file, reasoning, trying to run timings, landed in my main context; I just got the grouped findings. This is how the course's own transcript-critique material is produced.

### Quick quiz

You ask an agent to "find everywhere the database URL is configured" in a large repo, and it does it with a sub-agent. What is the main benefit?

 - A. The sub-agent is a smarter model
 - B. The search results (dozens of files) stay out of your main context; you just get the answer
 - C. It avoids all permission prompts
 - D. It's always faster

**B.** The point is context hygiene: the noisy intermediate work happens elsewhere, and your main session only gets the conclusion. It's often also faster when work runs in parallel, but that's a bonus, not the main reason.

## Fine-tuning (the specialist option)

Everything so far changes the agent's *context*. **Fine-tuning** changes the *model* itself: you train it further on a dataset of your own examples so its behaviour shifts.

Before reaching for it, climb this ladder, and stop as soon as something works:

1. **Prompt** more precisely (a better spec, clearer instructions)
2. **Show examples** in the prompt (a few input→output pairs)
3. **Give it context and tools** (project memory, skills, docs, search, the right data)
4. **Fine-tune** only if all of the above plateau

Most problems are solved by rungs 1–3, which are faster to change and easy to undo. Fine-tuning is a real project: you need a good dataset and a way to evaluate the result.

### What fine-tuning is good (and bad) at

 - **Good at:** a consistent style or output format; a narrow, repeated task; making a small, cheap model do one job well enough to replace a big expensive one (lower cost and latency at scale).
 - **Bad at:** adding fresh or private *facts*. Training does not reliably "teach" new knowledge, and can make the model forget other things. For facts, put them in the context (search, RAG, a file the agent reads) instead.

On open-weights models you can do this yourself, usually with a cheap method like **LoRA** that trains a small add-on rather than the whole model. Closed providers offer fine-tuning as a paid service.

### Quick quiz

You want an agent that always answers using your company's up-to-the-minute internal docs. Best first approach?

 - A. Fine-tune a model on the docs
 - B. Give the agent a tool to search the docs and read the relevant ones into context
 - C. Paste all the docs into every prompt
 - D. Wait for the next model with a later knowledge cutoff

**B.** The docs change, and fine-tuning bakes in a snapshot (and doesn't reliably teach facts anyway). Retrieval keeps them current and is far cheaper to maintain. C works for a handful of docs but blows up the context and the cost; D never catches your *internal* docs at all.

# Cost and data constraints

Two practical limits on using agents for real work: what it costs, and where your data is allowed to go.

## What a session costs

You pay per token, split into input and output, with input usually much cheaper. From the session we traced in Lecture 9:

In [ ]:
import json

spec = [json.loads(line) for line in open('../data/claude_code_session_spec.jsonl')][-1]
usage = spec['usage']
total_in = usage['input_tokens'] + usage['cache_creation_input_tokens'] + usage['cache_read_input_tokens']
print(f"input tokens:  {total_in:,}  (most of it the same prefix, re-sent each turn)")
print(f"output tokens: {usage['output_tokens']:,}")
print(f"cost:          ${spec['total_cost_usd']:.2f} for {spec['num_turns']} turns")

A few dollars for a real task. The levers on that bill:

 - **Caching.** Re-sending an identical prefix is billed at a fraction of the price (roughly 10% or less). It made the trace above several times cheaper, for free.
 - **Keep sessions tight.** The quadratic growth from Lecture 15's preview means a rambling session costs far more than a focused one. Start fresh often.
 - **Right-size the model.** Use a small, cheap model for easy steps and a strong one only where it earns it.
 - **Batch** non-urgent work, where providers offer a discount for slower turnaround.

## Where your data can go

Sending data to a closed model's API means sending it to that company. Often fine; sometimes not:

 - **Sensitive data:** personal, medical, or anything under a data-use agreement or IRB protocol.
 - **Licensed or confidential data:** you may not have the right to share it with a third party.
 - **Regulated settings:** HIPAA, FERPA, export controls, and so on.

Your options, roughly in order of how much the data moves:

 - **Keep it local.** Run an open-weights model on your own machine or cluster (Lecture 8). Nothing leaves. You trade some capability for control.
 - **Use a provider with the right agreement** (an enterprise tier that contractually won't train on your data, or a region-locked deployment).
 - **Don't send the raw data at all.** Send a schema, a sample, or summary statistics, and let the agent write code that *you* run against the real data. This is often enough, and it's the default posture for the project.

### Quick quiz

Your team has patient records you can't send off-site, but you'd still like agent help writing the analysis. What's the most practical approach?

 - A. Give up on using an agent
 - B. Share the real data with a cloud model; it's probably fine
 - C. Give the agent the schema and a few synthetic rows; have it write code you run on the real data yourself
 - D. Email the data to yourself first

**C.** The agent needs the *shape* of the data to write correct code, not the sensitive rows themselves. You run the code where the data lives. If you need the model to see real values, that's when a local open-weights model earns its keep.

## Reproducibility and an audit trail

Agent sessions are not reproducible the way a script is: sampling is random, and models change and get retired. So keep the durable artifacts, not the conversation:

 - **Commit early and often**, so every change the agent makes is on the record and reversible.
 - **Keep the code and its tests**, which re-run deterministically, rather than relying on re-running a chat.
 - **Log what was used:** the model, the prompt or spec, and roughly what it did. The project's agent-use log is exactly this.
 - **Pin versions** of your libraries, so "it worked" stays true.

## Summary, and the end of Module B

Today:
 - Example midterm questions (broadcasting, a silent `groupby` drop, the cost of long chats)
 - Personalizing the harness: project memory, skills, sub-agents, and when (rarely) to fine-tune
 - Cost: you pay per token; caching, tight sessions, and right-sizing the model are the levers
 - Data: keep sensitive data local or send only its shape; keep code, tests and logs for reproducibility

**Module B in one line:** the model is only half of it; the harness, the context you give it, and the checks you put around it are the other half, and the half you control.

**Monday: the midterm** (Modules A and B, closed-AI, in class). After that, Module C: what the agent's code actually does to your time and memory as the data grows.

# Appendix: a bit more on cost

*(Reference, not covered in class.)*

### Rough price shape (late 2026)

Prices change often, so treat these as orders of magnitude, not quotes:

 - A small/cheap model: cents per million tokens.
 - A frontier model: single to low-double-digit dollars per million input tokens, and a few times that for output.
 - **Reasoning / "thinking" tokens** are billed as output, and you often can't see them, so a model that thinks a lot before answering costs more than its reply length suggests.
 - **Cache writes** cost a little more than a normal input token; **cache reads** cost a small fraction. Long, stable prefixes (system prompt, tools, a big file you keep referring to) are where caching pays off most.

### Estimating before you run

For a batch job, estimate first: (tokens per item) × (number of items) × (price per token), and remember the conversation grows within each item if it's multi-turn. A ten-cent task run over 100,000 rows is a $10,000 job. Measure on a few items, then multiply.

### Latency

Cost isn't the only budget. Output is generated one token at a time, so a long answer is a slow answer, and a reasoning model is slower still. For interactive use that's fine; for something in a request path, a smaller model or a shorter output may matter more than price.